In [ ]:
import json, numpy as np, pandas as pd
from datetime import datetime, timezone

PATH = "arData/position.json"

def to_iso_ms(ts_ms):
    try:
        return datetime.fromtimestamp(ts_ms/1000, tz=timezone.utc).isoformat()
    except Exception:
        return None

with open(PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

def extract_confirm_window_metrics(rec, pre_window_ms=5000, min_samples=5):
    metrics = rec.get("metrics", {})
    interactions = metrics.get("interactions", rec.get("interactions", [])) or []
    dev_orients = metrics.get("deviceOrientations", [])
    confirm_clicks = [int(ev["timestamp"]) for ev in interactions
                      if isinstance(ev, dict)
                      and (ev.get("type") or "").lower() == "click"
                      and ev.get("elementId") == "confirm-placement-btn"
                      and isinstance(ev.get("timestamp"), (int, float))]
    if not confirm_clicks:
        return None
    t_confirm = max(confirm_clicks)
    t0 = t_confirm - pre_window_ms
    t1 = t_confirm

    betas = [float(d["beta"]) for d in dev_orients
             if isinstance(d, dict)
             and isinstance(d.get("timestamp"), (int, float))
             and isinstance(d.get("beta"), (int, float))
             and t0 <= int(d["timestamp"]) <= t1]

    if len(betas) < min_samples:
        return None

    w = np.array(betas, dtype=float)
    abs_w = np.abs(w)
    dev90 = np.abs(w - 90.0)

    # Interquartilsabstand (IQR) als Streuungsmaß
    q25_abs, q75_abs = np.percentile(abs_w, [25, 75])
    q25_dev, q75_dev = np.percentile(dev90, [25, 75])

    return {
        "confirm_click_iso": to_iso_ms(t_confirm),
        "median_abs_beta": float(np.median(abs_w)),
        "median_dev_from_90": float(np.median(dev90)),
        "spread_abs_beta": float(q75_abs - q25_abs),        # IQR(|β|)
        "spread_dev_from_90": float(q75_dev - q25_dev),     # IQR(|β-90|)
        "n": int(len(w))
    }

rows = []
for i, rec in enumerate(data):
    m = extract_confirm_window_metrics(rec, pre_window_ms=5000, min_samples=5)
    if m:
        rows.append({"X": i, **m})
df = pd.DataFrame(rows)
df.head()


In [ ]:
summary = {
    "Durchschnitt – Median |β| [°]": df["median_abs_beta"].mean(),
    "Durchschnitt – Median |β-90| [°]": df["median_dev_from_90"].mean(),
    "Durchschnitt – Streuung |β| (IQR) [°]": df["spread_abs_beta"].mean(),
    "Durchschnitt – Streuung |β-90| (IQR) [°]": df["spread_dev_from_90"].mean(),
    "Teilnehmer (n)": len(df)
}
pd.DataFrame([{k: round(v, 2) if isinstance(v, (int, float)) else v for k, v in summary.items()}])


In [ ]:

import matplotlib.pyplot as plt

plt.figure()
plt.hist(df["median_abs_beta"].dropna(), bins=15)
plt.xlabel("Optisch ansprechender Winkel – Median |β| [°]")
plt.ylabel("Häufigkeit")
plt.title("Verteilung (Median |β|)")
plt.show()

In [ ]:

import matplotlib.pyplot as plt

plt.figure()
plt.boxplot(df["median_abs_beta"].dropna(), vert=True)
plt.ylabel("Median |β| [°]")
plt.title("Boxplot (Median |β|)")
plt.show()